<a href="https://colab.research.google.com/github/Nurdaylight/An-Econ-771/blob/main/PS3/PS3_optim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

from scipy.stats import norm
import numpy as np
import torch
import matplotlib.pyplot as plt
import math

In [2]:
# This is for the timing of the code run time, please do not modify
import time
start_time = time.time()

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#Params

M = 5000
n = 20
beta_true = torch.tensor([1.0, 2.0, 3.0], device=device)  # (β1, β2, β3)

#>--------------------------------------------------------

# Generate tensor of all data with dimensions M*N*K
# Create one part and duplicate it to ensure they are identical
X_part = torch.randn(M, n, 3, device=device)
X = torch.stack([X_part, X_part], dim=0)
X[1,:, :, 0] = X[1,:, :, 0] * (X[1,:, :, 1] + X[1,:, :, 2])
Y= X@beta_true +1.0
#Replace constant for regressions
X[:,:, :, 0] = 1.0

In [4]:
# Compute estimates
XtX = X.transpose(2, 3) @ X      # shape: (2, M, 3, 3)
XtX_inv = torch.linalg.inv(XtX)  # shape: (2, M, 3, 3)

beta_hat= XtX_inv @ X.transpose(2, 3)@Y.view(2,M, n, 1)

#All estimates for homoskedastic are in beta_hat[0] shaped [5000, 3, 1] M*n*1

In [5]:
#Comute standard errors

# ---- extract diagonal using torch ONLY ----
XtX_inv_diag = torch.diagonal(XtX_inv, dim1=2, dim2=3)   # (2, M, 3)


##

# ---- standard errors ----
se = torch.sqrt(1 * XtX_inv_diag)
diff= (beta_hat - beta_true.view(1,-1,1)).squeeze(-1)

# The z scores are:
z= diff/se

In [6]:
#Сюда нужно плот сделать

In [7]:
#Computing t scores:
error_hat= Y.view(2,M,n,1) - (X@beta_hat)
var=error_hat.transpose(2,3)@error_hat /(n-3)
se_hat=torch.sqrt(var.squeeze(-1)   * XtX_inv_diag)
t=diff /se_hat

In [8]:
t.shape

torch.Size([2, 5000, 3])

In [9]:
crit=  2.110 #Analytical threshold (GOOGLED)
false_pos = (torch.sum((t > crit) | (t < -crit), dim=1))/M
print(f"rate of false positives is {false_pos.tolist()}")

rate of false positives is [[0.05179999768733978, 0.05179999768733978, 0.04879999905824661], [0.04479999840259552, 0.15219999849796295, 0.14739999175071716]]


In [10]:
#F

t_0=beta_hat.squeeze(-1) /se_hat
false_pos_0 = (torch.sum((t_0 > crit) | (t_0 < -crit), dim=1))/M
print(f"Rate of two sided null rejection is {false_pos_0.tolist()}")

Rate of two sided null rejection is [[0.9777999520301819, 1.0, 1.0], [0.8447999954223633, 0.9914000034332275, 0.9989999532699585]]
